# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Turbo**: forma con 5 pasos + FlashVDM, textura turbo y Blender en dos procesos a la vez (archivos y videos). Barra con **porcentaje y tiempo restante** y **monitor en vivo** de GPU/VRAM/RAM/CPU/temperatura que se pone **en rojo** si algo se satura o hay cola.
* **Ves el modelo antes**: el estudio se abre en cuanto está el GLB animado, mientras los videos siguen renderizándose. El visor usa un **GLB ligero** (Draco + JPEG) que carga mucho más rápido en el celular. La GPU se **precalienta** al cargar y los tiempos aprendidos se guardan entre sesiones.
* **Esqueleto detallado (66 huesos)**: dedos de las manos (5 × 3 falanges, nombres de Mixamo), dedos de los pies, ojos y mandíbula, colocados midiendo la malla. Animaciones nuevas **Manos** y **Hablar**; los dedos se relajan o cierran en las demás.
* **Por etapas, de más a menos**: primero la malla completa de la IA en alta resolución (`malla_alta.glb`, pestaña *Alta*), luego se reduce a las caras pedidas para la textura y el esqueleto.
* **Física de telas**: túnicas, capas, mangas y faldas se liberan de piernas y brazos y cuelgan de la cadera o la espalda, para que no se rasguen al animar. Los vértices duplicados se sueldan y los pesos se calculan con hasta 3 intentos.
* **Huesos de tela con movimiento secundario**: en túnicas, faldas y capas se crean 4 cadenas de huesos (frente, atrás, izquierda, derecha) que se balancean con retraso e inercia en cada animación. Queda grabado en GLB, FBX y .blend.
* **Piezas con nombre**: Cabeza, Torso, Brazo izq., Mano der., Dedos pie izq.…
* **Malla en cuadrados limpia** (como *Smart Mesh* de Tripo): retopología QuadriFlow (o rejilla de voxeles si falla), UV nuevas y **color + relieve horneados** desde la malla original para no perder detalle. Con esqueleto y animaciones (`modelo_quads.glb`).
* **Detecta qué es**: persona, animal de 4 patas, **cabeza/busto** (mandíbula, ojos y cuello: Hablar, Mirar, Asentir, Negar, Sorpresa) u **objeto** (Flotar, Girar, Rebotar, Bailar, Saludar).
* **Texto a 3D** (como Tripo): escribe una descripción y se crea la imagen con HunyuanDiT y luego el modelo.
* **Textura rápida** (opción en Calidad, activa con Turbo): hornea a 1024 en lugar de 2048.
* **Dos motores de forma**: Hunyuan3D-2 (Turbo, segundos) o **TripoSG** del equipo de Tripo AI (más detalle). Blender **4.5 LTS**.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden (la primera vez tardan unos minutos).
3. **Opcional pero recomendado:** ejecuta la **3b**. Genera un modelo de ejemplo sin página web y muestra los tiempos. Si esa celda termina con "LISTO", el motor funciona.
4. Abre la **5. Página web** (o la 4 dentro del notebook). Verás tres pestañas que se llenan en orden: **1 · Malla → 2 · Textura → 3 · Estudio animado**.
5. Para no perder tus modelos, usa la **6. Guardar en Drive**.

**Para actualizar a un notebook nuevo sin reiniciar:** ejecuta la celda **1** (cambia el código y conserva los modelos cargados) y después la **5**.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vttu4zYQfddXEMpD5F1Bli9xXBcq4FyaBkiaIE2AAkEgMBJtayOJXJLyJi32u/reL+sMScXybpLdfakAWxQ5PDNnZjgc3/ePmGaZpqSk5OTyhjzh4Gp+DgNWFksG/0QwuShKUlFFaKk5+dgwktF7FnneIa9JU1MyH8QxyRmZxuTkAPYOJvs4gBnEUow0inagzud/np5fzDzNc65ICb+K5wzfIArwOa1Bg1xSXGZ1a5yxcs3+Cs0WwRRXHmwQkhnh2omiyiCj2YqhAYIuixpMf0/6OVv31arqWZYVLUsKUNTT7FE3khrOZF3kjBNFSwAzCiXoKZusAKrOBTTyfN/3ikpwqQkY4UZq1eii9NpPoezn1cXFNUlALmL1upC8jpZMB/78cnSU4pofEr+f8VqzWvepGOV+z7u4uU6PTq9e2wbLX+1KweYipwp2H5wd/350/Oput7yFcA98cybbN4B4Xs4WJF2KJujNPAKPlk92gI8jqbnMVpvJhZ2IsianUaFSuqZFSQG0xWgfAcZ1RMG4FMJTZCwVkkOa6IKpIO5t7ZEMwlQTEdW0YiG8Nde0TCtWcflE+mT47t0oNjvYY8aEJsfmBZHb6BZUKa8D5h9e3oAj4ih2hCWt0rxQD0FV1OlCMpYu75Opsx7CfkilYHBgILW0pPf0A8e0Mzm3STBVkBWFdFKCQuL8jN81DwngZjzC3PnKnagJXGJTKEIDUjgzSxb4Larfi4xQl6fzuZn/JSEdk7e9nQP0M5DNFme8vyUH6VLRB5YXUgV5CG4slE75Q3ItG/ZiLEAIFX6v03ObkoLqVfSBF3WA+Q9p2BpjdXzbio32TeByU8naPLMpsoZogkqbxHanmbBHM4KDoZvnDAp6NqG2PJyJRlmjYZRmvKlBBeGSDM3yJy4fQohalboAPqcPHCAXHWMERGc/nll37Nh6aYtln/wGH173XOQF8LClMvFtrYQU5ZkGFcneYAgKNRMq2YtDkq2a+kEl4xifcCtENq9oxlQy2DPLBCodiP40CcGDeMyTYTyehlDTypTX6CXj4xdg+GJRcponv9JSoV+xSKYSkeMp4CpaiRK+huPQuIRJlYysw+Ea2bhgNJ11HDBGB7zOfH523eE9RkO/4j38Bu/h/897+ALvYex4n43JcPwm6/Pjo9MO7dF03NIeb2hP3mQ93ZC2PF8jbVn9AOv9YYf0YPICacVmX4DtkOsxGey9Sfpg/sfpYYf1cG/Ssh5tWE/fIj1+hfQgxuD8AGmTCd/DeeCO+A7BNqhqshVt251lQ6VpYrgooGeAu6Lb5MB9AbcOVEXoOWruMNa8XDOJLQdjEqVA1l4Y0Jdgl8NkVcBVbZoQVy8N8oG9sR0MCCBuCYpI4JZM94Ua36N9IegEwyoBaPBuu6teZBDErY8lzLRPqTFY+Xe2rmEiT+JWCroudFf6iRXLld4SGsdt6XMTcAw7l8Gt7zxo9lT0MehOhcQVDhE1IqeaBRi0TTnH61hCGc4D/ArJoBeSrel2djvCWMcT/LPxgxotk+3ivUFoZxBm674Rz1eNymRxzwLhbhu3Hix87FP/Fre7YPTu3WcS4NiZjd8nBz3y7z8mTXClu4Dz27fxwkcZNBolMIAg4lpoXLFDXIN5e3LMvB26+S8hV1RB94Ji5tzs3s3Cz9hsQ1aBdNsK4zqMEUM8vgTT6X1QtvUpbIBL/D/sVZwi", "engine.py": "eNrNfW1z2za34Hf9Clx2OiVbmZHsJE11q846tZNm10m8Sdo++/jxcCgRkhhTJEtSspVM7t+63+8v2/MCkOCbrKS5O+tJbL4AB8DBeccBaFnWy6RIsolYwB87d8TRL2LtR5EvTs6E/dsm3m38+OTs6Fg8EMfrLb0v5F2xyXy8fBrJOJCZOxj8nicikCKT802WJ/lkIMT34l0SJLmI4P86CST+/WsjAz8Wcz9b+vhOxgLaen75u/DxYis/CDuV2SKMxMvTf7x4+Vo8AEhCnF68e+0MhZ+HIk7Eyt8JmUM5HyEUmRSb2KchiB2CycPlJoQX0qVuXEBDqYReiVyKFLqIrceq7TenL4U99+crif1P/WUYA1S4vAjjzZ0jIrmDMSbUCxyBn81X4Za7nvqZH8G4JkIBzaAjaqyi8LMAuiKXG6iec0/OI1Fk/sx/D6/lOk2gutj6COpBILcP8tVa2EGYzxN8BD1z1AAYRdT7pJBLKeZJjE3N/TgAPP67nghh/wrlfqCqUCYDzMiYuq67SjVNFImljAGR8zCJq9mo8JbJfBMVNFfQ/HKDg6KCiHFGG3dVwGhgYvIV4i7Ss1Gh7f0mL2SOmEplEWZYZB3ma1VQ5iKM88KP4Z9M3IFlWYMQEVRgh4GmMuiru9gA4UFRmKD5Qr9fzsurKJnpa+xJFJa3YaKv3udJrK+TXF9lUl/lq00RRuXdriySb2aA/bnMyyfFKpN+EMbL8kG4LuF8CFOgYjnQ9/Fmne6w53E6WGTJWly+uBDq3Yu1v5RD/vM6zQdcwBUVBhbhcjD47fzNuZhCr93UL1ZuEGaxv5a2vvdnOf61PQ8b9jzHGcBUehevf/1fUKvsrHuRzG9sZ/DHi/M/38IL24LW4sIaCmvmz2/wbyQXdJ+Fy1VhOYPB4BvxFgg5C4HaJtAtH4kmF8sQ+AhmVSRi4Uc5Di4RKbA4kESxWc+AH2QEdJNtwyDJBjQ+F/jae/Hy9Pm5d/niH+cX2IXHD73RaIT/B/j27YszHOfx6OETaDqQC5H7C+lRs3a4XjoTIiugkT/8CLoETB8nGYit8INPdMkdxBnDt3bhr/04GSJjwu8FFi0Sx0UaQzjhAqoDieRMgXNqQ82GS79Vg/iT+SHwAbS7kedZlmS2BTytZAKOHpBh9GBL/QMMYtXboVjBsAC4m4cfpG77VvwsTo4Fcg9d7WvrooQsUUatQR4Ca4JwA7kKrdrrMA7XCYJL7xzVLDXxPQD/RXRPwBc0uMxA8MiqgbV/Z+PwHGhEz2AFFkasxj1P0p3tmC/cYgWEEvthZNu65rCE4ehpuDh99es/X78tG8SaKGd55mIg4jfPnyLJwp9Ty+luO97KrLBVESqRSRAoMb5WhDaPpB9vUi/ZFOmmyO0bKdPpkxHQThh7i0xKbzmbjkcVBT4FIeuThDMkJaBJbEP5HoVmSNqqRJ3ilcIXJOdLIiyyXdXpIIc+58D6MrADJFgR4CBRvrn4q2T590kY2ywe3Ne/v/POXrwBHHxvOQ4iSZcKcxAVduBUiMeRYBMk60CQ5DfeJkfuqsNyXCoIqv/7709GZe3bFcgX7CQQgbBB89hBjjOPyEI6pko/mygzJoTGB20H0LkktUdO7Y3qUbYGpS7tYCjCJbC29CSSYz59l21kvYIxTHkX5jBlAeg/ywXxazVaxR8om8l1spVmsVqpL0eNvJvLtBDn9AfUadV66oPSYAL7C7Ru4UXJMrcrIno9R8oR/jbM2UZCSQIibw7iSICJUJIWyLIEZI0L7IcKA+AmyJYEJwgXmxyaBZZ59ew1auHV7iQA1nVI+RNYibCWgKYsEeuNzNF0yhOwCuCFv0XZV0lF1j3Q0yVqOOPRrZ/F8ChnoaZuXNA5hcz0rW3xxAFLrkFjAv6mlvu90Xn3ey0V7wUw98HkSbLd9Bmp/z+5BNdG3kAlyDJADRjlgLp085WfdjySjWdg0+ITg2LUwN2lLC7gUmY2tuO4OdzLrYxsXeDP0zevXrx67rTZmLQ4TAvMi8xyFwkqb+AVDYIArivWhxtswwNZNUvysNgx7SupeQ+NdXcBTM44R7W3rxeF2Yvib/ViMJhH8FecxwBc8kukfc8D9VR4np3LaDFEW3YxfZXEQMjQ3DTNwrgwJgALuSmwIpZDsaIYMJAFMIahRaggQICi5hjoMTD7DF9gM/U3RAX9b7z1ds/LYm/FntepDwPsf+PdhDEKRsV/5cv3yQzVwUfLnxfoeFgTAfrIkrE3TyIf7z7VK3jg4oD2uYVKz8Aqq9r6BgxU9DpAQckgRF1k4wUY4sk2jJSYAKEAGh+wDY5foXRVHb4EUYX9IUMK2n/iYn+UVwj3J4/owYxdEnjweOS2+ghAPJTahj3brcxUl1202w1RXaNys2vuJg1AYNhYHqjCD+wkBQ1Vb9UxdGE/MdfYqmyDnR5AwKcmqfnBroVzeqVQ4aVJEkGJ+cJ9R9b4Jdyf34HLDILRBhvKu02yG2DS6QkYbEwZV5Z6Zl1XXQZ0rP0bmLosbyGMtKCX3ChdWTGfjHOQnoQIg8tA3L8NUfzPkyBcJujkAZ1t0JZWvqLhve/80nMHsZys0whpAd6U4PwiC2cwHPAENhK9ZKKpGN1d8DJJB2USpMA89MkFzCWqrFLpaOOAZ8sLwjmIi+oNiiQYiw/K0LYUJ4IUR4Zy9pZCnrynYMWDqBesvmLIjVBgHzf2VdV8CdWJRvrKAU6phc/jr95WNc0DzM/htD54RP7Yv94WgQTS5B5sF8dhqwSYcgGqW9sy+cUCUzankg0b8qpe7PrvMNaAxePR3/hhzijZDeYahY93K9GHzpXKyyQgeArDl2BqxcUDI8Bm0Ry3Hq+3MBU1dr0As7MoA2tlOAqDVxTjIo4L5Hs/i8oA25vTlzUeI7NgtSHlv/Dn0lttZtomyOc+EDPOMvBfVlUBmx/10NV1vwxGayxDU6wOw3ZcGnjbHIcpz+idF5KH01OshC23VMpFRkLB0VO26u8P0GGwcu2FSzERkv0EasHNbel5ft2rDdA0kh2aBiwN23pF8jLdgGEeSUkCrYwk/vZsAlPa8FaUvzkaVGwBtJIFNnWs3kwMyB7VntyGoC9Jm2FxjM/MoInZZrGQYDstpyMHe7to44QdNlQI3fiaQVOswOzHD8XPP4vjhltmzBc63LN+tKsRxp0FYpwP9BdnToUBNIkwcObiL8Oso9F2c/T4MY1U3tX7USSFj/o136xteQcqMrUzcCFphp22vWgvrEvmGBW9VQFpYJaJ+MjAlH83cceLT+L5U3xttYe2sD4aIxBHopi4o8Wn3NAiCi8E1VDKrJ7SMJWGhBhi5GoooMIscRq2fNOJ0Vxbioyz8N2zKLl96RcgGOLlJYCOQsPgxLYAR/eVd7E11FjgPMBtYDfoWPVykUQggafUX/ByPAzUgRTL2846kA4NqD5l34hn4Cqs/jgDlAcSbZBFOKelAF6F+I/x6I7CKZmfhioK3yt+9PBcGfugG7wFgt4Ga3s99/xomUyt9bzh7St2f7dLOeLVBtjZSm9LbbY5QKC0BYso0YK2UxDmaRKH0FCHSDkQ/mfDVgSLo6zolexpJtRbcA/ArqJZVndkQpkPaMLNB3+hYVnsyOura7VfkQFNY9MV77C2sB+pMMcPZb8dDEaCyYrxXFp2gK6BS2iQFq0/gPUkcj+iBYpyGUb8CijCxRQOYYrCX89CKEALU9q6BeAlKLWawwFm6M2Iu6O9JdWLXynQG9SUrF4UwKBHJe0oBIKBdrBdcne+CXwQVMV6E7lA8MmtVyxOjoE/EWl7KsXx5xafgW2xAsfuplm6JlbYX1Y9fwoQllkCiHtDMbPMZGZzPrsttFqJaWl2Qb+9JPaW6QZsNgzfJIVBLffrBWjbcO07W665/q1R1GFVTVM4s+bk9wMnTiKaBaoS5jqpolomP5JavuO6bkPudEqVRoCBnSBDQ3Sbjb2quO/HWqnqwVEQFkfb46PRESEABEA7wPoF8oVRcJ/g6iCRGl02aaw+O180NzwrGFEkbu2Yl1po6MAJ6EJo28fTw1lvaST9ZEfxoy8YHQBWtLf1s5BWIPLCz7+A+FQE66Dhg3/y+RTYSYTr7delwgoh+yixMS12jeRAOvVMjNOetS+cs/ps7aPN+6ftv3fSuqasTed/Y7Zq8zRE12qTo5KlFYoxJQr0zB6HVssp4dvO+ahpO476t0zoS6zeMpypxxuKxdpXBi6oMVOKXjfUigT8gNvsgAXTXc+6bnnROfrjtIKd78HboouoqPfC/vgdtf4ddqa8JIjcoe/YovnuUwel7bV7e1+0At3dCN3vWbSIrlvk1lyPbl/1ftPeoCKLp05haLK3S7RK/lUx0DlIZ7AfvF4vgP52O/XgNt98LXfkEGW+f6HDNN1K/kTbr82j1SoBLkFx4BNcy/ADO8loRMJDMyDeauDKShYL9Fas68k9HdU+HBn63jzdeKpqw5lDrrnH7lGJcS9O6/gS9hwkWIYLWTtc0w0SI0fKsWriDDHSZSm3A20prXnW5P+wqa7MB05n+C01p4EkaPnENEd7XGC3SGwLUGY53WshNYOO3tz62XqT2p3RmJdqmSECVcihmHvDKrq2t5bofJeOqmqmvdjxe+yjA+cHmH4Wh2Cjg1fHuXsTWtIII3ZDb2QWw5TjTP36+9mp2AEBhEtprA/nAoMKWVisOfw635y9esU5igAjzaDXmc6D8zHNz8e4LSY5Rsn8MG9RBWtqjkHNKDFJhworpacdK3YFycsMc8/f+jA8oEi7QQzs7N/rerUDvl2JbGeZf1tn+zXA4zyeWN6qBJwh+NLj46GAXw5cj4ZC/XOcVq5ICdbFX3a4bhRxZRSFaS5te/zj46F4CGBOTuDieDRyKP4XTW24Horxj/jrEfw6fvSo2RBQLSZcLCOE8xNUH/+EgMYI8ccK0GN4iBlB48edYNDlZKw/p8RKjFcGchvO5dTCebAc8PdjMDBBuIHobwRaKejJ1cN4ITPUCySZbGifn/ubIpn7OS7FILihCApQcVN+iYKrGD92ugNjNuVyTcP1ENMRvaqFvJBpPoXZSOaYeONlEmyuDeqH6fj4SY+tiCDmYM/c5FMtsfjWuh6qrFLo1HQJUCmbyqN+4vLQWuarpnFbEWq+i+erDETnB2k7gx7DB0M4YCqD0kRWPkRUfFZkX2jga0ySTUSyDosw0OZnKWZQN3lFolITy7SGdUqSVwbTUV36oIZIMHFaJdVRxjRPQui74tnF7/9wx0f5fAWiJxJAs8cPgfg44jQEuTingiXE05RWG45xPRDjWaXz8O+CE6Ul+BRDbYWche9c8VaytEOxh9Z1lBwghWhEKIeKzOYbx81xmc92riYPR6PKfl1sIozAq1KYYgWICJGnxHwFtvy8kBnKpmT2HphtyOVnSbADxwezkTOJAXvMR8UEutthR7TdSiNQ27imUUgxK4M70E6yAKVVbIIwAVQtVwD+5AzwS4nRebFDn2IFzwHrBcjBut5dwrMChsdzaK5PdvsRnVZwPeFHI/NZtLnr9CZ6/MMmFRRENDoRs8dcV5rCbKttZs4iQNcRGA9AF0egCPIH9ba+zJVnavFMMTTTcmifhbbN/LW3nIG/9MtUPB7tibKTlbEpE2nvt8qa8fn7bLtyJmDeOSUa8AgEgDD+frgdcNzt2xqc2WNTN91V6B9SQcNlhfr7CUwPS3mf1rCjZrcfBiKLXJMjVeFBVfFoOwbSOdPkDld5ARpSBmbyADopw5odA30xGY+9mKlglE/u1aVsbtZ0KBjxNl44B+vS9oQZyx02CiUQqBsQpaQcURVMRyhku7QmSOgVLfJPxyiu93LQbRgUKy6HOdPQfUzcBlCga5bw5vjR45rqdFya7vzKELFmT00x2ylKWVZWUtJSeslAWW1xx6PkjXypVyOTpJhaD+Ygj9E9fYcv3z63GkqNnwr7j9O378QDQffi9IUzUYswoOUo43kRAheifoInOTzkkM79+qdDPJspJv2OpEJVfXGkKbgxBwGGqTPJ8x0nyXTw9E7nU8c5JpKPGD/OIQVrmTdYCwaRz2EURW45TtumVtPgpoo58/JKz5BGlcJ+J/8TqFkWgjNkrOrA/ZuXT5/XeaWZh6f6WGkPncnSFTCnLnmZWm9R8FvqpwYfVTuWOhq7DwEDhoh35daP7L5WyKmsjfj+dkqypWbaBoWyo+sWdI9hqImdfFRhr3H3HBJ4Sekl9VvOvbR4qEmqR94KqLT7tVfPnJz156Rw2mIpDJS3WZMGtC2GhB5LkUpG1jZcMAfLNe9/2s/Z6iG7BPUlSZJ8ntpypUvCNKd+pvYC1VNFdIvuK38tg3e8vS7bPYNHdr5ZLMK7qeWm8RI9JhnJQk45ck05Mw12pz06/hZUwcqltO9GTijlrap3/ZIlDSNKXza6rDJ3ZkuPQkLTOHX9LPN39tUY9Yv+dQ3eHrKTF8ti2mAxYxdHGPtR1Gi02uFAOUWDL/RLO3Tqwfo0KVU9dlo5nYCMhlfYpUyxWdxMpKjsGLzrvRq1oaSJeW39kBSHvv5FPOLA94/uCARB7mP2apdiVcTovuO/dgJlXPC3wbq1YbqoiROMWeDc5agaw+UmQV8K5zG5Gl+bSrU7hIlO0Qc5VM5JMx2iFSbt02xG18nJmQrEn7rRcSKwtPmJGTDtjuTU10Vc9GRwdoqQjGiGIjMX0aLuNF++hEdv6MmgJ8qq0k1NVCCZ6I1w/TW4IV0h29tIiYeqP7bKOTVjGlxsKMzOTPFXh9THyUPb2Zww52+nC84x7YS3R/LuRFQdKsW3GWTQ4c1GMLNtyCgRmdMG1o43ddmLP1uMzHEFdxtmmPeNTSXZzu6LanHGCpXxMAUFd+eAO6Uz5SjPqy/oyAw46ovp4C7iDYVmP27XLjgVQS3/7gE+baTljTgt77/+systj1PzMFL0cakgKI74Tvme311jdeuzNgN8jVxhvYPWTBdOTS3LItybLTmLriYbcEvjHqFQtxKwfGNja15kTkvdlRFayi/FTbaNLZSNLbiDjrqv0xw34S082nAEhkOj6MoHaojSla92ZNIGTnT9KCBMiw+lPsSKV67rDsXJtbsGQ84RP4M6GJkDK5HEVZOiaqFrfFVWkN3cD2ra4Pt3jPLkZ3IODAX8C7SIoSw6eAC9rgx5mrfmb/wgw6Ue3ESx9rOlrMQ1IqBzpGWJHSi/u5yLxUn8QWaJ7YMOG49qUSvMpb3LG7N5BzbE3XgodvB3NwYYdzljcEhXoF3halc+26lnHRgjJGRJatsIc0dwwdkcE1z4axoHHZubib2VUqp2CH8PBs74ibGE81f36gAamb0rA/lfIPHBSGCKtu1cHIlbEEIgGLAq3K34zmllNEJVUAWoiW12xPF3a6MxKWFC+PFDFlv5X4Z1nEZh4eUrKQutz/Ea/LwM9TnvqB/iNvoh+t5DtZG+5jafhVvwh2m3+Cp5T74D54DwyRKxxPz1HG1HSkzGPMdFEgcNf3nBLe8VCMZOpWw58+us0Me5jW251EZro7waNUoURpIhQ+gdWFp9fFZjAuxXyQWE9rLcnPZM+a4f78zVklwugUKgJ4BE7IGfFbQ7QSWkDA01g6ul4VDQ3gEJVieaoLifN3JakYBtKUoYcseaaaNFzm4NG0vFClUMbj+o1pYxc4SunwIuAyBwbA7acWr83wW5qxrKCRyvUftPQiq5eWCy1tCKeMw5DYUiIvgICQ3KAWOhzQwz9Ccw8sgdPbxuyiMsTsJ6QqISiEWsNiAv8wmmzOJpEWAvg7Get3Y7YG1iIJf4y7aGzdiI7h25JyH04E8wBWJnSCIGHqBMKp/yRo+QNnrglmM7dqrOoudGoBK9kkGlEyzd7ENVK9kU9Z1/ev8xCB0Wuw4C+ADwqIEh9bjpHm2KK3x7XSbFovJHDjCErZLh9BSFFbkVrWQrxeAA8auYJRTEqFwXdFxDHXNDo5+DlrhhKD90X4J9JpXTJ59BPZWCiwPBm2Azh5sDFx2eoeclMwWiQgL2jJY+zNe2Y+PzVqnO7rQK99rWOywH1mzt5RoJqXzlou/xVhYNjboGpgwCj1zKellq20UZCc54zph2GcktEPgujJeeMn2CDRApWuCerm7vqYKz6YE14YHDEWL6lCdhzvZUmUegCbxVAsyKCpwu0FU6Ho1Ge9rxkgxXTZlQvHmyIicyavoU6HusMT2dDvdhzDidfksjPGRMZ9NZXxMe5R0MEV6RpbMmXBoPVEhp2uClz1wfLuewJ+SmetYRcOtcOtwzVD1cwDBYkX0zzZAZGzzV7iK88zjLL2/Qd2+FKNLT3SrfiZ/j/UtfFPris6V4BZ2d2+OaEitlCilLVAIGB4hfqveTLgowBIli40pKga6fVhKrKTUJ0Y2tWErY4dpzXsY6OSmiGfNUG7TwWCFweOsuGoeF0bVREVT2br5sFaOTXtTozcwgFaulvl+pQ52u+wK2LYDf6Lh6uWcmAWNQMCvTFD6QGGTzJ7RHHCPvPiZMlamnfUtqPCVtE+ezV1J1/xYYdrg3tK2t/QpBqKfsz5hZpxkNJwCfTSAtD74rGLPeRAUu0X68mYDRiFbFjTJYqSU3LOQaOJ5NVINSKktEZ7GzGUXwkHnGlR0NFMjJy3uSBOHBdblVT2Uu708xvHbEv5pWtO6LynxudIH25Bp0a4Lc03it5eK6mSMXy7vCTqvMSG6rmeFY0QfuViR0cdrcvamUhsjKaXtEV71iX7UF1WxoFCQg1IPgi1PIHe9BRz1x2LzXT8TPU/CJGCePvixfNClWMmt0uI7TRufbY+nMIFVwG1mktaeIqu4kDSrWkUV6X+qHEWKU67TY8WZzuxnFYbaCTqrZJvzVBeT/mwWSv5G8pyR9LX9vz+qJoqBem7qd4Ye1+Kkz3JvUN/iiVRinvuzT37HOJMGOBRpWKDyjX8XtUXH3WlafPipFOz5K4KNpuW/pxrLUYgsedIkYJtsnxHSMGEM8kVglGWjOIHE5tDQtj1PlvYKt6M5BC0GdSvazF4WgObs5QEf827Rpt+CSiDZbOgzk9pJXC6jJpuSF13nyGrfK8KOba9aGfMBXzwGVpBZJT+LxITfOniMqvhF/8H6rKu8yF/Yio6NQf8BTZf0IZsF1HcF53GBu40kTIkqEH4ANK5u+fOnF+xTnpIko+TTXURG8Jn3MMghvgbS/2PvQRIPB5doGMjo5LmBbe8g7p3hwHY7IvZ2v9bCKNfAGva/Deona1I98dyN3dfMqSQsz0IBiUJ3rilppbIj7kkboVNM63lDJm9TRjvnt3Ymy0idK3bhALiTNnf5CW9BYs12B7tnVZCJ++tHA4Kp2NlWwWae5jUMc0lmPHgxfHZrQ0ZDC/spdybsgBK+9wAbGx9eVkZptYoU+Dj9T2pc+YA0YRl0Sz6hritHydTve6+0NZjfXqIZtu0Y1wrpF3ZArpq6BiHU/MCdtfHzSSJ/TKmRKJx5Bu1NrtcHV0jDAI4h02p37BDOxQERxll598qFoMrV+9cHC8jNryA9Q/emmORFAdwmQXOBiZceYlEhT5xlkMgrlFlt8Yhr6FF5YZhhfiPz1LPAFqpCJUKiIPYyBhn7UgXTqmQpLUFHg56XGG+YsR7pb7F9OjXROlLCsQFRYY9I8b41P/6oIyjyIstOgpD08nRKolj57WYqQQdOoLc0RVDuXVxZd184xI8pQJdUNF+UbsyxHCLgoX3NJuq6lNtB6PRbDKy7U2C5W0oAqWN1z8fLerKQIRY9L3amR8V2tG6PGJhoznZ4IxAZiPQYl9r83YUGJ37S8Y5ndBKHFR+2RXpwY4WLF4iUPOvr4ZrMQsatZpm18sSo1KxGj1wEzx5uFMs6vr0p96lyS6paqpnvLLoa5nKaWlCoJ5Gjvt9P9MCV7t4lUQycaFKAOBk0Ipe3RY2a1TkZ+Rimt+qBzzAAkNWtMH4p2POg2nCs2mNbCBVMdNWCRSL9ZJsL/Zpput9lcyscq5IBCEv6XwpH/lPKR/wx78IOikn6bYrK8qmSl+tsNphKhsySJ7PLWKfNt1Au+cSpRyn6DunN6oYOk1ZBniTPUcpOe8bVTyUt6qm4cLTjpdw/8SpDqXqp7M6ldyzy0WExbpZbYXpI3nwlGXnHtRObqJVDllQWPQID02XxvyjOAZcynjKFFd9OZy1kDXAme5uHWTmNda+/xgCTK8iJbkDizvv0/366/Dbxvf/v25bdvPQs3t990H1BJBmPjVMrKzqDUIzoWvNY6HnWPxyrydh53Vawjy6GNZEZlVI/YbaUpURbbSunymlwwaQR98Mi/5kAv+SRAj9A/7BsFFukdBq0CdsXrJp9pa24567SewUwtLyyy772P8SdKYnUMhXKuyVEaR7V+I95g8J1kUwS+5kh8y6cgoSeD53Di9ykwEQ1NFD+X+vMbeKB/FiaZcZoQ0sb5lTqPEn0z+/yqPI+SI3bMyyq/C4nhvDyekWo8rHDl6eNVcPUYXo3cn0b4aYyqgQfYplGBNbquWNa4txcMp3bUXhHOb2xwjGYoYOe42JZTRlSSNuYK3PDTrR9/IBE/wyQFkW9gbtbkKAaY6kNSnz48wujLlnXvnZA37tlKWx3LRwYXdABz5XJcNmyTxZ2KEMLAHw3BR6/vdBzjSClplkY07nBPSpPDx9mbQS0f18nvaPwdGxiwO7d+WECLj5w6BlPwwGQbhYu4KbkARu2zE+dbQFoDA5RHXRXho/7sArOliinO1VDAdT7tmTLgcB8skLjjXHjgHsp9sA9Ye1NScxE3CnfmV5f4yVtLvKpd4loDa+UpzlflMbHXGAQfm9pCf6kDhP38BhMqG5mHhs14AoLxPMa8GH+iv54DPIsODq7ZJOrIMqsRmNQtdCiYZueOzM41SulzbxtD6EVuvFUnMtZEH4lMQ15u0bLKuzxrsqH6OEjHRykcGKh4PhEo4AlYRcmMoSFdDs02sDgKjHZ5eZrcyZmF/GM1j1iyyHHaVuEdq+f0jvYPgfuv/+STiqzG6geAObS7rPcm5vk/+9a6vmifZ93obKx/dsxcKekbAivvl0/lhFkX4ToNCf2YWZnxTPjLzXuZJbnVtX4JcpAPjSxPppYxbzwzQrE2Rpww7VrZhOokjhenXRAv6QwHytrOaIkZ7YucW6EjH8YglqMooY3ccx8/R0VffZp1rIZ+I/7j0UhQch+vime49wS/DJKnG8r0U8+rwy93Pu5jBiJ0O1LHFolHY5s2V8y7nKSq+C/iZIQ/X3C2z948H2MZvhfAgSv23MG9h/occnxNOxcBmiQaoHO4/VpGgrN3gV1nQnFf6yblvaijqZF3iKW6YUeGsUVTTXPjLqOZ5Xy1E0SZhmjS94/0G3HsGCSuCRIJMVdUneL3AjBQTgSqg8/4ETawJIHVEudelia7bURC2aLZD/Gzasg4H0vqnAw/QbMfOdkDrqlxq5Oi22kiHSkiX4VyD6TaHubrnX026y0/m4ME6Z37Xgepq3rDP6r5STYTmsVlzMkHOZj6YjypyR1cJpWfr4srI3zS513jywB6caPMelbXNSIZD9noJ7VdWvifqbOsyzAutPo2l0tenH7uMQc13aoXB5vLgn9HpeofWhFkHKqgA9sBJHE+5+yDDjz3peGwpuYy9+ppDXcfRbP1qRYAww+41PmFxN0HaT+da4KpU7qm8+MJEQHtvjxchvduoDo4H+GwDUa1jUaHuSJdlnnNfjcca/S5H9V97R/UI2WsDdqczIErIqN68zX32wBdPayAw7OBEZrQn6tcbGTGy+LqS5ZoYJFDAzKIswbqH6ukE9NcA9IZnd+OCZ1JXn5DdAJuORh5RKN++YmEIZp95CDxKjR+LZDjhRXAYtZ36nCcNgNH91Cp0cnL6ozq8nNb6lxpSg3gmZtQrjz3SMShXr/HA8h2czNfngidls0tHiMuhVM9joMoEJxApYtc934kyAycNgppI1kV0nHUaoUix3AyL1Hgt0/owyTgn8O1hZ0KQHDhNXeuPGKB+l+tGMgWlE/1mINEY8JW0Yo6EXJdoGeiQYyCGE+G4s4Z9BglKPHA+Rq5P/0Iso7uMSRSq03BkVpPtv092bZ6st3TE/0OkUXl86XJIBccowMuoJCjRIUMchKsVvrg5gQpSAVH+YObwp9lYVYRlgFLxvpjTjpKEPmaOdAedr9OOLb8qoZds0r2cAjgv9iAp9Y+kqEZ7eQIa7m+3ww1ZfNu3lzc961AqNkRbjPRACWa4yxPsjG/P+PmmxlY2XY7bKJK4R6VdMjpE8YCjV6ZUcsYxsJLudzSWIXW6ynluglOjBYDxCn1Ctp6sZV+LG12pSHNVWmW9eUtuijmnqFF65toeHSqYu6SsZsJN/gZYSWSXr2mZTKfPkpDX25eol0F4l3GxjH9eGxNjI5/w1b5RvhLrEIfRfzni0ui66Ck67juJy+2B0/Uf9dk9U4Y42zI8qTpUvLHdgkPrNlQB2iVWem410//51C8fXeBxyvJD3h2NH8AaCcuXp9Vc3b3/yO5FmSgcWpDffAz/pLfQrr81ct9x09S2R+mwvpX/K8YQ3OLu3atz8qs0limHjb85cKbRW0jeWYaWtWSh2EPmesg2h6aRf2jKrOD7POh6Ph4HSDw1nI+f8+6jhvVd7CFLbuG4xIw6ZhfmjQ/vNcWomFz7UQ10vgCHxQzoKR3+5v1aAYOafyu2fpdV9t3Df+DhmZ+kUxbifxpM52pFd7xF8nqb512OoPHmi9KAo+jFnh4ErElXv218YNcvehOY2AA4V2P/wG9vbrxkKzCO7y4Zyz0IVlo+OraAUQVuBW0Bdf3qOe+aliN06iJ3YK3ymijRlplnJ6zZxDhNaLTCCUcULBnimdHRLntMNQyCISGBeVtkwkwVYF9nTzge6E/ZW9kWH7VxSNvZkonnNmFDiGPe4LnZS0FsaxHMKFWVUKxblliFh0CFo+o0FXqwmLE8NtJDJ0Q/fcbMP3yKecIpjJbhNH08sriKzplNd3gA8r4r/BdCRBEqpIhYcpyAwU7Ll/iaa+cFOb5+TwMdd6ZIqIp7uk2Ullwa26bWxV+UHYiZITv3mYhzDOKZqcr5emnJ7h8tU59sEI4PgOK3GDxDySPNkX5eeaGS4J7a6nxD2kHfDpAybrAw7lqH98qE3GCMJty9TCdfkiHRJxTxtON3E1vtEpb0MeGp4utfkAG+5Tt9oZXAN55mK88NnJsqOkcsINrqxTdpmjrrEpMbu8Rk9RmU0wajHePHN4nWLdOjzDadgHdOk6f6DpEzm7vl7O9EA+Rdt1Sb3uA1LtP6BndurJKIcC0QIaAbkW/61kbNIXntl2kxtRfg6cN5rgi6cudjRftr/SU3x08QASA/O46Ua1UlislIEqbbVuTFAcw+xdtHuT56FgB6Y2saQ+dYhjXzcPsG76i+mA9u51d36w3TZjFniygxZa2iWOjYC9EEaZw2jX5Yhh+rfjvob2p4UcnitHpWnnf0fqNZAksSc/+B6IpnK9lsUqCajetOXPGmVVrlO4g4lG4w41VT1GASnSC3z/DlI7tgxJE4MPqzYtL7+z82cXpu/Mz/gQxiv0IP8g+HRPhfWjnmlJUAlPs+XBIPLoR0+/IuW3POwLFThotvn33+s35GU2di59Uw87atuWu04do3alTBS0d72AfCkz0xs5Ic946R9Qq/UExS5vvFngefYUB3jaFd7X0tUyCJz1XiEQeUns2fgsL+mZBhIeLmd7l0S/C39J5eaCqXcAoLWwHvvZEqf7LFxfngJJX52/Rlaj2PqkoxERcVatNlIah3aqJOgxRL8twoW7rpx6ocPFTARUYsowICsXBdLk+SND5u135hVwC98QEB4w230QEj0v0AiIHO9OH8iCkk/r4qAB/wKIXytpPOX1AecQIBr9ZUIHRL3pRw9G7QOf6eD5+yiIgTD3CDp1zARzR84unQr8+GN6tnClqdvEzBxYCoVyEhL97CO8PBwbGiruY3RGwRueePf3HAXBKR8r9Ecngkm7pRKTVBhB+CAQ98T/ixL/ElaCSHA4eBzh3COLJYwDxB28Lk7lK0ciFffH6zDkYmJIR6shP65TtHoMVLcdY8VTxog7O+gNf1PhK5nMZ93BVJwUfG3BKXmIa7oOiliaIrDCETIyA43ijV1awJ9RpOv25LLcf3jIE+jp5PKKJaoEr3/YA4R5rjP7BEUfNigqZn6rtUspw0afufWm0y4hwdZhWeo/Q4yEHR6e4SdvikKveJ1RuDaIdN80dpfM1fgHlSqVjP704f3V2jl9rPyJVc3S08OdFku2OKNtxk9JDWhAa0eVle7G7K7KvY4PIqumODLmjI24gxNkl9MANYIjDxi2o8BIqWwp/cMcotOh0MJtvGCzjVL3gG6cLXBkjZdzrJ7QvhyuX9wxY78Lhl+quE3Q5ZzjEsU6740cq9Y7Rd1SJaOpt394EKKkWxyqIzV2zVQEywqp2OoARsViKaKgjar2uAs857/82LfVuubm3XNvTLVThIjqqiApVSpy8E9VQ41W1eliZFXQcef1D8GnG3yFTJ+u4l+QsAOXiAAKglqn58sXlOT0Hi9R8/vbd2evf392XRoExG7UJbrZZ0GlE46pvaDbXPpuO5h92GC3ANHO5N62sfOuUjksTi8xfS1rJoDqqLicSs9FnPcOwcIfVuCbrGfxS9GDtrAZR2P8KfnCsIUHr9L3X5cKLEgr8LUPe4DAtxX+331jVsjFJnHLE3RMwsWlLr4sH/6f22MEs8Yf0QaQOKd3hk9LJxnHD5eGz6NXpcoScTB2f1l5qUwKNv6vZb2KY6O63HXSpDtTrhux6WlO1QLrz71kjbULq+XZdixiuUOI5zbnr+C4MH0Bo38jdUCfd1w8jXOd9wRZwXvD7KIE+WANgaFz05z0qPqAD8foPXDAIp7kTAHiF8v8Ntk+WxR3uBAHH3WLdQcRQC85DNfYPcfsyyqZRz0ILQ7s6wlzPSXuTOb8eDAbe+avnL16d6wkZDFBzg7zyZLzEb5ZAwWmaAaEr9C2jZOZHQlUbqG5pKK2tfRX48wpeGY9QndHA/i/mIQkI", "blender_rig.py": "...", "app.py": "eNrVfdtuI0mW2Lu+IpyFWSbVyRRJSVUlVrEW6robXd1yVfXMwmyBE8wMkllKZmblhZKKEOAXL2DAxgB+WWBhwzuAYWOxaGAX+2DD+9b1J/0D9if4nBMReWOSoqp71nAPRpWXyBMRJ849ThwahvE6SEU85Z/YpZgw82XMXS9sM1f4TCRpBjcDuAnSOKRnQZiKSRhesJA54SJkEZ95Ad+LsonvORyeBfj/NOaJCDjjDK6WIoFPWRawNAsABFw/9cPMnfo8FrZhGHveIgrjlM38cKKvPyRhoK/DRF/FPHDDRX4n9FUinFikebMkm0Rx6Igkf5LOYwETC2b5A28h9vKOadKMJ3C1tzeNwwWzmXoHs5l6M/VQBDBboV/NRDqWT/b2Tr99/eYdG7KR8ZQvACOxYTHjaRjHgq7ecT9zub5M5dXX3PPl1VsRhUmIV294ECZ48YpP8OX53tnb5++ev0fYqz0G/92DNVhMYjFgZsSTMLFY6KSxEBZzOKDdYqm4SrOYt23W6zJqAou1EAAXYCxCWMWQvc/iSUjgDLo0ANqxxQ4fHlnsfhf+s1i/e/Swbck2z0TKfZ+71O6hxY57fYv1jtcbvuWRJ1v1AVz/+L7FjtZbnQIGsM1RV3X5cL3NG37lLajVcbfW4VH35D40u9l7/henb86+ef5u/Oz1W5ibcYCkB7R68CoLrjMeHD7r9A94kgBpHIgrvoh8MQagM5EYe3t7rpiy8dTzRWK67QH1GsauiGkVJZ7GPID2bmjP/Akuinr6MeNuUnsG3FN7Enszezq5Kp7YE18EbqmFXCnvU94DDSL/TzcDPuSp54R2OPlQ+jp/vEj90mMgzlgkXhgcunaCb2pAI0984vnoIwH0oW947HiwzPoWJjD2gmloIzMCJeLnUySiEXwS8XRufwi9wHQtNm2zaRizKfMChUNvynQjceUlaWKuf9POQX4FMKcFiAS4S7gmygMb/6x9a+wb7Xa7PDHob4rCCT8HgCAn6G4KLOsml146N03D9sOZYeluQWBkMTbWpJBkiwWPr02csqIHDpMlDACrmwb/kCWpQO5c3cjOfeB9Qkg+FHMKlAs4RCkHf1+fDtj+/gphjFrEn2Nk/9b5wLqR/Lq/z37+y/8Io3Ezx3NRZBo4maLX4iujzYSfCGbAxVd5j1NDdUDNZScti7X+vNUud/LT/9CCga24bAr3rfYNi67wpVECKJKPmfBFGrISYKAGbG2WHs0zpB7orAsv5I3FiKIYCMgKyNJXkfqo1QF4bey79DIRswxkfDJOQxA5ah43LCmR8dR4EcYLPmCr1iLzU28J5MVbFaSVXyiktXpAX2nYuqkMy2zqeorgqVca3XuFNlzLypRaiVfrVmF47BXdBthlYzeqcdFRGfhboFsRDyoLQI804ioI+fkP/8DeewLUUjJgNHy2eWIFTiuTYSZpgnZ9Svgwnw3MpUxIW2aV91Kf2dc+zaPx44mfz7H545XZWnquAAJrsa9YksZmAxTZotUGLoFmCTMFiBT5kkU+KFjCNsypMtG1z/MpGxVeI8LchWT399UaSXlTYWrSKw4IaRAnStbk8sTmUQRoMI3TohFQHkwGhbKUgSRQKlDOlTx0KiLL8XniTcEyw1ZGWw/EKbpMvYgMg5UxzxaAG5g8KF3QCzEIfY49OqDt4iwSpNdln2TEHYFxkfKkrmBIyTh8AloG28srdsAmIDzJxAEtBqIF36mrmxwAiKoZjWZ0QdrgAsW5iSorDjiZRZOYf5IGkgMiyieAchi0nA5N+0KJ+DWUTtGSEU4KClearyA6CQH0maMoHh602har3LZvYDnXJ4r/fYVCn5lLtJRXuPxyieRcQMKRPFcz0/K7vU4SZFSMSVxvJAmtXICgaVVgHgm0WEQeL+mZEqSqopHWuQ+IvQaF43tiKdiGSRnzMA4EwZeaLAT5D3atz8zf162g35PQT/nnf+Bs/ymOK0z220bDHP3QvWWGxjffPYMnMxGHa0RfGSnIggsUrMt8goBoohmLLZFsJIsUPZ7bXioWidluQv6Cg6E8VjjZPLi3ssGAYXtETIASFSxI5qB/gS5SItGlxGATEpYiBrNNoKj0+ZalfvH5xwR9KgCILROlDEatCgC5wsvPfyufUevP/yuADw8cGOQB8PQMHJtNy+x7ExEXywyKW/IaUIhkNWBykPRgD8BcQYoSIWciaJqYtABum9YraqVnlU+q9HHrHBYWcViaB3Y94SC8HSG9TAAK1ipvJDOyMHA1QZ3g3LaM5iy3VhyOjBGhDcz9fFh1UBLdURakOAdv+vlHBz2Ijfg1U4+cshmPkURZGoKNCcMHPxhBIxqSLGFL4Xjo+G2cjEeODYq7Xps9Yb2dJwQ2r1xfQZ0qONXpqYeEd3Ccr67Z4vMfgaZFjJ+WUFzQbuIFY/wYUIMEgkjWTLd9aAPwSrvsN0qqLEicJR6QFOMYcdCG5wZ8aoutxATrI8F56Ncsf6qngTLiUo9VaWOwFEmpjM43U8rPf/03//t//oGtLm8UIOVAGD8EPwRKQtEnbXAo3r5+SR67cZpJVcczqfxekZLFJ4W6xUhB7HGSF4OKvi1Uq/G0qkdzzYpff1fXqBhmgBnm6MRXgReAnYLaVvo7S09cing8Txe+6Q6/BSPCYnPhzebp8D562fhePb70YjF8wUF3WUykwI3q+SKZ0ZVCmmEYz2XIiB0+YybGXAT4jm0kvQzwPY35AshdxGBhlUQliOcIoFJ4CTxXsNzRH8a4kKI6dORcUEB0cYtXaaiglY3zAj1bLKdaLljJx663BNvx2hfDlpzxYCX/vYmuHrleAkbi9WDqi6tHHFRR0CHFMXAERsoerVOmgX6hN73uqPCDbokQOi7gzknB+hqA5s0WwaMZjwa9LnQ04c7FLA7Bdhzc6/V7h72HTbAn5E93MEKVJYPeEXxIKnxw74SfuJw/mkKnnSlfeP71ILkGB3XRybzWkwZQ5XnTV4n3SQyO7kdXrSf/57/89//w+AAaPMFWT1DxmrC8iHXjXTbB8B0nBwoUQ5T5CWdNHUw0eDVE8VCcCNF68lKAtOYxkgXK7tenAP65Js7HB5Mn6NHC93IA9FdzK8bdstgHPQULHSd58O1jFqaCmnwERpsa6RAEQ2piXM/GP2ab7aOoAdc0JyQk6YIgPmLYYWr82XK4whdFMyT3WjPjz4gHenkbYoO1RpI5EHk0PJPu5USS2KFxHshg45hH3gEGn4arOgm3yiQMnsSfrz6qsZVIWDNT7AxbK/h709Kov/TcdA4U1v3No3XiltQ06NYorYnK1smzBSLaDy+BeDLfT8DiEcEjdhV3Ehg+6MxOGsMnIGVaNeDys+IjWGE5/BqRVjkTLagO8d8gxuE/AsUJFmgnDaPBEZLsY85Awkzz+afwXqTDFniRPLho1QjUqFLmw8nDyXT6qGCD3iHCPJ3EXozyKuIBhTzRTYh8FE8//+VfPT7gBXXu7d1jnV/wH5sppkBdTYSSAMQ3XgJuCTnqwGtodsEYQAexELw4TzpeeXQTfP0skP4f6FAcJ6p5h7toqxA4UyrYzpPcZ4fLXC20bfYUGhJvu2IJD5dojhRSGJgdvlYx/ntgS7AFmGmwzqisQcRRpBvgcTkBm6HWgddBsvAwBImYXHrLkAUeNEfGpWAnwLf3/uV3X5OK1PpofMm91ExSiwVAGuBSxGC5iCQpKxfwSjFChmgBZQEdTfiHkIFtRvYDjpXGwX6PIH7PFiGOFd1+aM2XaDzmeuVyDsxHXemwYZKODLwHVyGPH+IzF/BrnBe8rgdmwuzNrn1yYlGzKbjgFmo1Zwi+CTxpLdAQwfBWWSh1sHEr7YIhaXenGIVpl3xxaJf4QkQA9zi3uhA6mGLlMcTcA1E4i+3ncRzG6Cwgj4HVisChLfRMJCoDm1FoIhkXmEzm4eVYz2McYsQV1jbxcMuHo+1EiwekJreDcOG5uwA1WKhlKYW9ANbESWXw/AJj8HXQyiykZjYYegHaOcKEoX+dpWkY2I7vORdtFO6Ae6CppCzqVuvwBmyEkwHUYoz5QrrUBRXhakmjRk33HvtezQ+tPNJBRJJELkvxaYDE4Qg/83msfBygLxet0oAtxAJolpeHVLab0AAynhPJKW5wgQneh0hxSfr5j0j/P/3dIRKBtozUjsJP/zQArCZsAio12Ow/KNl1KgPzwI5nof/5x1mImzmKFYPcji5WXBlHFGWXWIhgbdYj8OV5RQ0x+6hNnKCfgnhFWYmPn7BjMOMJ92gBqo7B+ojGNBYTFHcAvIxaxGK+mMI1iXFglrkQ+p8x6R14BXheivFkZhGMYlOrjpYpB6Oe9rigpRAwC2TwsbycZZ6LTE5daWsWbsKUpwItWleEaxDpKTgPADWhLSIEDwhJ+cQXejcNQUrfH0gwvwYLmF8IFTBcA+zSppkYvo8zbAkLHQ+NXHYzM/78R9wsaxsk6xZROjQM6m8c4y4aVxZ3HazmhSGw0JkWRe2SlCRy7AF58QAchpKcvC6EfO56yUgSeoS05KJsd+eYLeQO3gElyX1YG/9Bowu37vb3D3sg3HqSpkQwg2bFBqmZCzOiC+YlRDZEXUQI+knRlTL8TYkbMkSNtg0U70VmyaxvkIY1gxV50Yk99cilm8jxPv9jYJdkby7Vu3a3p+Q4uGWC1AcgivYNA+W1IFKBLsKf/81/K4vv+Lo6LjnXIWLDxvbjNJT7j2pSkoQLAOLKETDV5/QPSmAO+nzrVKfGtySxogwjmjDaeONYQTmIG6N5HYo+ykMGP8hLx7Q85jrTtqXvTB8osB88JA61GW+n4YUIxnNxZd5XJnBKLvEURDggGXcs0S89kx564JLbCksAD6V7SUovvyE1PGBrTKF0NqiFc2zllT5Ju3BdUr4y0IyGxwiGeo5jTfOYQOi7qKp84AITm7RHg06/W9K5+NCOwsiElhZ1AfJWMvqUyAdkHvB5iTaVWYDb7vzKVHcgd/2Qp9C4tIOJ7xayZQEzDMagjoFapEHk1iDjVPELt/JUWTA6boG3pWFehvFFmXvWSBZBxGQCSRqIs+CLRfk2laY6K4t6LdvJjRNXbS3gc6l+KzyMuhBVkviHP+1NSqAk9W8f5RatcOvHSlMM5ZqrO9qqKPSJVKPIEreCo924IWKIcNdmj4fg5BbG8hAvbgcjNdZwEoZgwxQqDNClaW6oL24HpvUcQVM37ds/UwoR5XmYzAxp6MYmPUY5z+NUbtwb77FJvvk9lxrU2AFZhSqlsRX31dyBCjflDKAe3UU469wi8IMFcstaJ2TFUxhglV5HwhRtezxGFh2Pb6R8XvuEgNFHOVhbuodg5TlmdSJRLEkj/6x4PcXNG7+B26V7A/DRTpGSIs+Wst/Tlalca5QeIIc4cHhAZo1aJzWKwoMzpD1acuKkBFL4LRCbRzSkiAGpbFUDkyoOacw9opCSZSv7oMQISljR9OH4/NpQMUsyvbbTiQrZKKvpgGz1fJMNgaM3HPoZGQxGja5BD2eRC3LFTMChdlLhDtXUaw1x47ihC1Q2fH+fmdtdRPDTz2BhtSmiXHk0Pqq90OKjRdEu2+OquUnI3c167A8KUxHsB5U0hqE7BSw3E0mnk2ZEuQtdlLzWdeuibqi9V5YpuF0ikHmDbviIqZ65DpFoO61EYHp/bXcSayQrAGNYVRroywV6rzfwbl/xtLlpseZVkHdY9VoI6ywMPEHhjHyf5JqVkgCkOVpaemL6uyz84aCAbZVBY0exMwfHOfl/vvZjnNd4vLb2cVl8S2qwaJcHPRZFEhZe4TPVYkkmawArY5rbstpUSho0wdUDUKZbUyLarq6oLmlmGNKNNmZeHB7e7xp5oho9rnjrbW1ZIrhP6LW7GBm3P3mRRDolLMJjc/QpOm9w2T9FShCOzjGmnic4buYF6LKJqlUUuk7V5QCLVb3bq5kYVkVcS1QoFNiL6Aggy9nU2ebn//Rv2f7+Gyl0EN1IIbdE0fb3aRsOZ1zJ46vCBr3lgXmhBqxZZcJTZ27KocjIgTISt/KM8k/ps81UTq7oksceWAvojepw3wYHGZYZxuCH+BL9mdF57p94lqREEWQLYAxYK+q63RCY9NgB2OmBapAHJV+EtPELOOrdHKyKBjdlNxhICQ0U2oNOvACIGUxvc4pxmVhR1tRGk2WzDxGXfAeER0gdNhnnhOch/a0yE+JBuzDAsEj959UWgKPS3ixQTL6PMqHccyATvGvfUHrl+st41AKorfPK7Hew8ar9/ud/v7Hfsr+t+E6ubmXDOJxpKlxcj0nPJmYhlF9yXwDpYCRW6uAEvCEeu7Shb1J+I6y6WEKTUAY7oOU0xu3HdhHJwd1Loqecltzbs2xl3rv93ffvMbtapdyid4fRYWlZtdeCNRqCl0gJWUUdCnkvyArK8RaznYQvylGY6lhO1Y6CmUECBZ+R5UiP2o2G3k6JyBWpqzMwFrOadY941GtvwmuLrS292x4NesfndiwiHwxw00AtZbA8W1kRAoEaDe53z9XihwBULr8plukABcg7ksTPeMoVGgGltIzur7mEextwpVeQNBQPrs3bkZiv162OWZ4svdu6qmw+VNSIgxFgyPYCV1yd7zXQ3p0TA/ShkXmWen4RS6Fb2wmj6yosfQNDoSUfk4odj9trfWDIRo5Q2iArGX+qyEv1CjPsbT8EVwspYX3w1VR8TU11cVVSBTxJ/v+0b/60JspuZkkF2l0NLatugWywOuZo2EAjRYmfkLfvys+Etmae1ioHxMxxLmbQlc1iR+Q6pkqLkhGipMIIOqgRgvmfP1wuYLSyob304jTj/pj2yq5LEQkU7vSZ7WQut+H9GA0exIpJIbDSSy8Z8yWeh5r4aN8RYnFfoVtSzk0GUzUtYqVQ5KqIvynsqH2D+vbt6RtQyMuFnSXCBfMI9yu6A7sHtuNBw+YbgIK2MtU7b4yG5suvMed0YUcixiyfm99QLvnLs+9ZlnCXD5phmbNR7xyM1tmoC8RS631FL+u9VBLfC1D51jwl9DncmQuYF9pywqa73J5ptGWu1vKhMOYmrpAox69ev3tPEfpUR9NnUaYvlzFf6OvSpVO0wA1+us73YmXkVNPaOnkhgSyRSFAORJfwp2t31+kSBWRxpI/MSiMAvvV4J1l4rNMBhzW+7sBghwja+8RxvjbcW5ImadH1NS2qtX2/FaeCNnYWCwIThSCLbDfml9CbjL4NnWRpBeFccIx6B2EGPJ5sD0omc+H7ajvQ4RFCH4dZGmWpeoiBBH0Jfg68Gx5iiM2FqxwwnkkayXDylRTgV3TsSm+NyW0bE6xMow3kpm8to5QUjy4O9BFZEusc2lmUl7kP10SMuGHBR318DOKFjw7P8e9RAQLpYZxepTKaqb7qdftHRNYHK/w4fyJp2rhFaZVAGrj5DlwlP7lV2iy5n5B14KQmkgHOD4ANl7g9DP/k/AqIh9cKFFyO1XNUDbjowzQqskQvxpbELYLX2ePFcIlfRhfjfK/lqhBJmMuYvx8NOifdSjAKAVr5fOFqgSuhHYGKuih8gZjjQY48pcY86VKM6G34IaQ9XDdk3J/hkdxEJlP6HqZsAPvM+TXm/RdRuwp7NQ8G0Jnz794OvpE+TlfJ4apkKraevJEzw5APZlnCHUh78pNUVtU21/hDOEnkc56izApgkQ18iOfxDO6kGJvCrUW0w4Ixzhfv1Em9b16/KRPIyQNFISd9SSInXUkaJ8eKEB4eqk1LcMDkGT/TQIJUUhH+uXc8OZ5Mu3j5GzT2TOO3oGUMLSuxxcnkwWQ61S326tavaq5b96Yn7v1JCd5T2Z+j+hNd3j3sboT2HuWvlsPY/mg6OXRP8PKnv3+q2f93FnsFs6EU3xMpbNHCAApw5uCeVuMO6H6mVddR7kUiawCGLZaR8Ug4KmgBIwg59RdifElna+dkFM5HHZAYT4a4MJVWmCUJUz0+enjkUsifPiN7ADosO2j4ouqh0XCrO5+l9qgk57XkeZp+irM29CkTkGa/AzmIMZSHJyTMrNUr0N2v4CFmfF2RpGyTbJM6fKsuUaGbq2roBqZ/0h2c10xsWogiuvA4Cv1rTClnEQwsTYatFYz1poUxJ3/YCsCuxazPOLwAVlthCuSWkUwN2bJD+aHQvm8fV9Hbs+/ftA6elPxooojSeJKIB5q1I+7iDtHgMLpiJ3lmaZ5B2q9mkG5M51+11FofHraqw2nd6/O+e3i/daMSnmGGOn+1B50moQ++zWbAziZwrScrpI0b9njyZEV0SApqld1gJvLjA5ylxgJKEXTPQMzIdActWXLLFA+dYaMnrFuQVYUQySz1+Q0ZbKGvzwT9SZG7jlga5AbE6qZrDVX+9k1TOjn0oUL4CWEyx1FLyWJ5LhhxisYsgcV2hIt1TIOyGhOtoz3xmK6uesNWt8Wu+kCtvwPqvob7FbJh1+7hbb9yqxlBzeX4WD/quDyZ4/Gb62HriB0hiVOPaLphIL0hbV1i48nPf/037B2dy3DD+mE0WmKVtU7564A6KS5/KDwW2pCsrKzqQYp66OGv/iDTBeVBMo2Svc0HFspp2f1uv9+frDNFdUHlqOpLX0+9biAxTYUPATA9wGxpSpRm+UGDR7VM/20Z3ZWTFfUzEzBxR3QmIr3EtHI6OnEZ82iAf+jMBB1TWOGi3ax10oBhbXOgTgbHT1k3lI8rn9RdQRABB3XAm51ATOuNLuXt79SylY2Y8uCWM9pt/Dq8AnpmXQbEzFavgGSxUIOIl+KUsmPfottSiPUNyfwP8OhIvQ+ZGo8oYl1Akualm1VLHUYlzYLMCIPZfY0U2murAR1osCTD2je1cxt7e0/fYWo3WpuT0L22mC0PPdBiczy5x1aV4wUPeg97LvsX0kPk4FbvNXwBLklH4eO4C1gotX/EKmR4s3dv7sZs3mOrIsG/jxTMFKa6qknEVtXTNLpBH1HJsFniuaI23EnP7R89Ymt8p7hLv8mZCXHINDfh2R8Ei8k1fwq4OFx74oeYrKV6kbfVzqTsqOBQQVcIkX1WFuUeZhWWoSCR8bhDa4Ve1EnXFTNLGcbWvQfOkTudtpsWajqdlh/vaWMpP47xoLbA6/Kp9vqqA1LeDS8HyGL9I3ivxnF8DCOPvVkCLpGNBMxWirBvmH7u8wn8XVU76VeJ7GZvGoYpUqJmEmTVSgOkeLWnmHm+O8487cBtcGrO1FZZpPJuxIKiXLFNl4n9NU8wWdTDcOJ4nomh4QWuN8NUyUBkaQyOMD1NfLAqt0cfELfDUQH6ZRjOfPECnpqyBhT6HEYu2FHTJTxIOmjc49kGOwHdnneAfD0uKGFMJqmhOBm+bXo/dnl8UWpUAEPybIAmaRShNTXQ4HSrOjhayzHRG7Yici5gld7mgFST6iQpcVdDkfJFz694pUHo97WxEHlVIUl5k48n9VJfNPdV2iuIsnQb0hsbbMY6nb8YJwJkrIv01bAATn96SIOst90+0ttA5xjfAf5G1Dpz4VwA3yvsNoxeL/n2lhuXv/ZZMyFsbFQbt+T3iaCkmVJBLbTQzkYUMhjPJgb5xIcPlekoa3PJ80peOkfB8DWSS2ISvQyN0zNGeRJ4qgW39/EAAbL3kP7C6JJkCMq4UTB8SIaG2R4+WbmhA35pkNpI1DaWDEm+8RLwX1zXbOEssNxMG+M9rliUjnfrIb0NL03hi8XYc4cGaFWjtu8LTd4AFJDN6F3fY/VB/xCsxe3PKtvylY4+YvRPHVSmZPBab3c/lYeH/fBo/KePmSdiN6yA030/pdO7ZuKAOT08tDAUMJau9GG/a7F8/qh/6wgow3nPJ4nZ8L7WxjTUmVljQ1uV8IFHI0hhYHIZkKFJZDg0vpMHHT7/OCln7oXMpLInlFHb3iGFttIb7mjPQx/k5tB4/sEeMM4WAhT/EuzjiwAXRM4g8Xx0u3m8COM79oF7Q0PjnUcHsTLgFspqsNiLb77/CzrfsPF4w/YTVBvi/NfMz8SMjgcePrPZC3maEqf1IaRjoGBY+Z//NrHvOA2qAzDs4/mgq7G8OWpvhKDPXMAqvqbTIWoNX1QSOpC1ryPg+cjz7zIcwhblFgyNty+/PjXyY/79w+7mQWlaPHUcUJWAFdN4g0UhaHceayqBy+O7HEu/xCp1ah5+4HiaCbewG1lzUx/I17c0JeEJcrsBTc/lSKr42TLtXn/LtPMVFNOmJTmjyaKosH/d/uhAxeYOged+xQ7liav1zl7BAiLByTWu9Xcn4GNdSRG6eBaHEcn+W5EwkqeJLDpjQsdNCC1oisoX8tgJvcE2u7CB+hK/KeDkHeAjuqnDN853OEXBfbS9N41ZS2LARKDKnSCHGHfiubxKAjPferOcu2rpVxsoSuL/LTrSJp1rwhw8TIKQAz+VNUnohK0cbPOJw63p81rvSUfK2E4bZUvAeJxgqvwTHAWeIcGyXEyVHVOpx+UyY9YdBLwqjXJApVKYuciw0NUkdABuiOHKayYbtFnGZMWUuwA3p36YAqiZF3PMvJjQ3QSTF9r24wM5q1sQoY7JyAV6qkxIU2795mJNr7ybG4vW3XWcUqnPhKrxhEVnEAORhyJboQMeumAqZD7/EjWq8XL/vioCSXuU4hN21z9s343gT33KkQdV79+B2KVwkuh854MJFptdG/TvscV69gN5qHgoT3Yq9C5EGofJ3cb2Bos04Kx0obA76LrSca8ti64KjNH0KbtzQ0GxW3qpIQJm3bcBFV37YY6KAhPo/AdgtnL3bth4CkPEj+6ABDrcVZZKow0HouWZr3cvgXnR6NCnynaRyk0iurkTNf83NCi0szAGf1cek+ylh4sFOPLzMnwC7oQ04eg9O319y/JRQDhdE9uqBHQuudGLzId/BrRChWnvLB20ZFDSZSCrQpOzIA+HH/f6IBx6x13W7XZlkT1AIPjo4C1iDvLaCZVdRQV1RIUXFgCSzuSBnL5f6udRieaxsovLVQnrL+uQ5kTfy1KBNpP1pQfsWNXItvRofKqztn2dYGA122ZkYKYLUi4WscZ/sUo1njeRS4ZpVmoZR7h856PD8/ZdF0ytuD5HZEZX7dsHqo481qSOquZVgyiZg3+RnskJ6hVVjgTnEDFSVgWAmIFayy/WMlQWW0gVhunwFGrB8jfglgcy/0W+Dj7/mHruVgn/BS5Ifrh5mxD/V5kHs8SAq3sbIak1wlSwbRD1+sjiWXcT07+V9XLNMCLf1rdkcT8i8zt5a6WDwluJSRf7Uk3fnB19KT0pgvoWM5gckM94HHNQErBYtzCWu83smn5rQLA0dJGJecIqNXb3bj0OXWNoKuifm8pFUX81SVWb9/M/BpRqRRBu40V9fHzbStNqoWkZMsyk3gxxkipDQpbRyeNFzTXWjFKEaiYCtPq9TzAvf7alCxJy21OSyxX4KSFZJuy3R4OH5xvhYvm0q8FtjsJzWbA/MbHUC0W2k+GIfK3zwkb+IKDNRiavxNPeeIkMO1brga71PF7D7KmDGYfeJ1p/ibdksQVvM3BcCMBLNFria+1evwcPhGVBKEuqciw15oeqLqdOZb8Dp8h6ghRmUr75MUYiqdiaPFIhLhVjbwQ6xwRUOdgXni+PlI2dMAtSsJCxhjqsQEH0//r1WekA0i5IP11iERoUg4O9X0EEU9GDinF7SNlYFqvqVkxnJTu394Uq9ox+NsP8+d/9114Xj8ZTZfRb+FuWsaiMrofphfeP1sbX0+M77BdxAvp8Sx9UHqMKXv8ehvy32kdf94G/mnEXNKjxPCVzD3QF2km7OEq7rJ8sHwSNv80WE5xB//Aox8A7sfB02QDhUAG54S3BprxE0VaNdOoLnmIdr21Mq6qMVBGMPzoCXiRWq1F+03EO9QU0IQ9IlhLDQPUW+LL+SAV6p/cQ4NIfDR/TULUa9VCaJx5beCgt0hi1tvnT37d/naXIC6DUDdkjHM8DlCS9Lo3v6Kibm7G4U6U/vLMzVqWvt0VZhZ3UpyrDUsHgkfqNGRiWem1oqi/ISvmpTP5+wt2Mp29CWDOzfH73TibTZKyPa98iYOkxRltBxxkU2TUKLfeidG54c087x/vq8aVbIf8i8+hW6Gvq9ozOeWAlK0D+rZ/Ls9LlwOJtX9xV421j6vXR/1bELJOFvTHP7IDJpPEdzIa4GNmGuXzB1qZIHAEr4+2wqfmgvIuJeUNNSht/cyhLaqPcUCF4f1/ZpPv7NgO8YHX52FuImKqY7OgCynoqltqoKwUFrrEGKRW4KIdc1O817WpH4S6uMp4eNKia6qYt7oCDAZ/ssHnby2u84E5NURVms7hYjLFwjUTsq/dvvjHX6kyefsw+/8g4GI/CEaANSnUfd7DE+uWiJnJMeUhx26jyUMeOg2osF7PTCJvKZKqR5gc9N48UR7ZhoA2dA6etiWbF8s+AZXiMv5lgAvfLWWCG7ld5ARSWgOCUqmAnKmsUMZuJrcF31+VZydfloN/YT39XdbN/+qedFNOOBsKSfvhM4obGYK7L9WIb+eEWI205Bje2AdBL7eFuB9MUhFeHiLA67dn3IGDlKZrbdgoWOvBO5FE9Y1USfItwk8aCN9tcQ31sq1nWfwgnynBBK9A0DFWTT+4FTbHCs389luFfeXKlVqgLJghWgsVQQWg7H9tVvWx9plO3xqhjWupKdmA7cx7MhFnu1GL631FzVT7g5PP2BkA+X0xczoIBA/QMddpSAbJUeK1ajww1Hh05Qna1SLhaShJaUvZYihYtSUlWwbmW0kV0vrpAgwlrJGsKm0U12kY+Hf2JStQWztqvWqe2arX/2sVqy7uSqlitLktbXr/zrZgEKi+WTy6mXiOc+DmeGQucLAZL3LkuYlF7jcGNoiGdY0TXA+vPiZhs/P19KmstwdcEvJ1kDp4UNsvlz9ToRoqqyqO7y7jYHXomElf9FuT+SzorelPSSFF6WZxZqvopPCsdxS81ltw6YGap4oxVlCNoawCjGY6eIkQlzoeHtqzAYBZFS/QnxSyLDyalrqnSEkCeaBYmt8XSPsY5vYLPLWnYb1iY0sn/Euy8toEeDNnT5SO5i1CW2PhyBBAMcJsyYYLU5qDLx7cRKmaAkULQeTNKRiOovV/+IwJU5DneG797/va3z99WC+pPQTqNMTfdxD/FSeKnHv6wDxZ1p0L6uM3sCN/lBAxNr5TqvU2yhOKVYM3TNqbvTeK1AvBJ6FyIooiu+sE2L8CkN9wFuxSTstWGcRo9Vqqle9FQecfF4nFYQqRaQG7t4HIOFCggTETNWdp8zrw4bqmKpKT1URlpFgQCExCKwWG9iuaBpbYq/p7qMSAqIkRFTPoRMWUxwtdX7KjbrqXQShza8h/pZiQ7TDuxJx5WIjK6Nv0PFW6DqavLuzch57t3VKZsHXalVJOGAONXpOXzLHDmY7nMNL3hg4f39cRoosM1+ss5CN4VxyMKvpJATToRFVNV1GFpauox9SXxKXtX9RLwhJ/YlL+EwXARpGNZ13SMGdT5Z+HlWCAO1AOd5dZriplSaF24YzpNNRzV67GUN0OUwNBUNdIsQQWbkfXXRQHxrJP/MrabHx25EpUfNy41MbYUQILPygWjq/UzQETNBCXWdT7S3+9wh+hqS+6XMU/TKBkcHMyAZLOJ7YSL0kjKgzoAg0PwRCQHeCglSQ8wRoCC92Czy2+Uvu8AorKrDl+4949QglKGfa1aFczVmYPsxllarBs+OD6uSFd4XCVVkEP4c9kJSGaX6uxnCaYxczwpr+m3V63sp5tXqyrhk7Ui51nsJ3wqzJP2PycDlGme8SydD02clZWPsyHBAJuNF0AGWOO56fjA7rzyi9hDKkbyjKW71D9eTIw606D2qHAMIBpu6Rcn1uSi5JMqB5VrIzrVMjJnVGlrRARUyHuj0wnCDv5am6w2JR9Bt7v4+lNiEuCRXv8BrVtvsMKlutklVC7LvAzLI3x99hxtVHA/4vLzd++fAVJL5WJKP0GA1XSKOpmFnqFfu6lW0EzZY3a/W6tuKE9gY0+q7oyNZOBXCy6RAYpFVQWYghgVMeNcPIx451O3c9I5/+oHG5anWI0fUGRgZBW7kL/mUC8+t1hXRHLBFzae14nMhggB2CS8KLS9qXK1NvlG9KsTqJ7HqJ5L8zzfUti6SpSKVs4VnnYpjiLrchtPc2TUSqCg8M2tDJzyWvVcBeH7RP+ckQjwOAbYZhOwfsPSb/XQAY9YpueHtl3+jQuSMnVbqVkgrQklkjVSEDSLms2So5H0765LC3KggdGIUPI2LNGa3FBqAVpLyV8MfO//AvZlo8w=", "estudio.html": "eNq1XFtv20iWfvevYJTuFplQ1M1ybMl0JnGcy8CZZOL09ACGkZTIksSYFzVJybIVAfML9mWB3X1bzMMusA8DLLAP87YPk38yv2B/wn6nqniTZHe6e7eBbpFVp06dOvdziu7De27kpNdTrk3SwD/aOaQfzWfh2K7xpHZ0OOHMPToMeMo0Z8LihKd2bZaOGvs1AIvhkAXcrs09fjWN4rSmOVGY8hBgV56bTmyXzz2HN8SL6YVe6jG/kTjM53abcKRe6vOjkySduV6kdZ8dNuXIzmGSXtNvP46idNloDMf9++1Ou9veHzQaUxZyH+/DttvZxbvvhbx/v8M6bncPr+ki7d/n+/yAc7wFM7wdsAOXMbwxx+nf7w17w1ELb9ElsIwO3L3haufBchgtGol344Xj/jCKXR43MLIinpjDyL1eBiwee2G/NZhwbzxJ++1W69vBkDmX4ziahW5/zmKdKDUGTuRHsXoHNcZgBLb027vThZZcJykPGjPPbLDp1OcNOWCe8XHEte9fmQkLk0bCY280iOY8HvnRVX/iuS4PVzv302i6nEYJ+BiFfTZMIn+W8gFGQcx0MfD5KJVPsaIQj66XTH123R/5fDEYs2l/H4PM98Zhw8POSd+BwHg8uGl4ocsX/d6AIBtXMUDpP6sdK+HjZQXNxqmFSIyB5Fu/TQeNfM/V5CwJKJtsxMz1Zomkbcpcl/jdnUrSOtOF3E4bztI0Ck1rmIbL0m5pDPZMWQySK1yGlPPdW2s7PSpthGetjV2kRLxwAkYD0yxOgGoaeYITVxMwpoFtHN4Po4IHiigrqpB0v+N0Rl22IfTKmj7Yx4Y+d5cR0Hrpdd/q9rJ9XT5iMz+lBdMl8bffxrPrFqKOuc9Sb87F8NE6U75eBFtodF3NCng4ywUcRiEfbCqZ1KiWVLb2Ns3/lTqwh+fAC6Wz6LcPaC5gi4aytr3WfFJYBJulkSDeiqY8XDvB0I+cS8zSoMZM+SvlUIUZqL3IkFO+SBvCLPpkRps2LABc7kQxE5yRfFLE75NeEcXVA9KZSnwSS3ItvUMFc+L7Ezpy5QhyaIsGZouSgPn+csM6wDeW9oUY4Un4YnqLHkWXuZjKOGiYKG5cZfJoAc0wAknBFp8kJ/rC1oRb6hGPyXpHURxIO4ZO8z/qDcwYJe9TcTTCFblezB2BHgTNgnCb98ocG2mMlOrB3vwKFLLQC5LlhhMkyVT93ODTLEm90XVDxTCFOUORKdCvtruSq8rUpLNmBz/potaIsrbRhVC3VZBiXA7cH41GQOWFo2iLCGU0QdASNr9HNG4qFWkEgiaXks6kuDugc2em27Z62CVJ2ZgPmTvmS8LXI2kpALLmkuQetYTkfD7mobu8zRWJDZWaPVpzFru9dWdRNkOVOTjOL/ZVubmvnb/iQ3Nm5Gex4N8015tXfG1x0g0Zy/HKiooCb7EDlaSAGK21yhB4y8zRFTzf4q26pRN4oZCgcqX34Tu2ScILkRPClVWo20LVdtPaVKeMZd3VjsPCOUvWffpvAu56TNMLZdnbxUGM5Y4mkqOMM6Rh+5nr2c8Ton3KL7SNBKNsegdVqXbFCmkihFTqktS67RuUI4k6Zwlfu8CnDeM1RdAyf6o26LYkdBpFfrKsurZcvRsLFQ21qqPKDtXboqvqVGsmWT5bu9eqHKqq6LRYKW0FvnrQskV29mCRxQl2xQmyVOigs9pZ7Rw2VdJ/mDixN001Kk3smhdQaRGwae1oqV6SWn9ZSycx57V+bZKm06TfbDpuaH1KXO5789gKedoMp0FTAP2mZbX3WlarOZx5vivHrCByZz7HipopUTXBryhMmj8PJV+wAGl80vyUBM3aaoVDCOJxiqYsn6h2wBuMWPNcuwZGo/LRNDHg+CxJ7Br0sSYmqZBKxDQApBw1l6WsMcdCvqhlC6KwdvQeajaL2WFTwh2twyNLBqOO3nr8hiU51FbUMqQTME+i5FaM2Py6dvQkdjzfv33fiQcYP70d4McZc7HXMX7iqELZYRNM2WAO+LV1wnUl03wivpZvo2Zh17Wjv//Ln7TT2Y3293/674KcEgrKl7ax26cCWBSltaw63X4cAlzAxL0oBqR60nS4R+MOlmMVfuB6HY5lT7LHW7dwYxYg6XRY7egZHr/8mZ5vhYZDCQH5ln40PUYJnrBYS6W6FGRJnt7JWYcFwXbG/s+//sO/acdf/oxww342dx271jJbtaPnVMbx7ecATHtfQJ2gEPNddivYgYA6ZW6keTc/WnewHcCNEjQin3Ur1m7PbO/XjrrN3VtBWuZ+i8wh9oa3UzdCEQldZ2VD/WnGIzvfzveTBfk/iPNOnmc4pAAq+2U/mTuiwC7oG7MQPIFX5H709z/9+zpcESpqmnDTIPZrCsZKWtirZGGqDVOkopsp8JY0uRrFBqJpM4E4r/otraW1KTNTmPf3c8dRHFcErc1xisebozK2Fe6axkSMvcUj5X5cBOw11afxOEorPvwJIm4Dg2zT9oX9pXAPxzCSuASwgfQK5RHMPfK//GUchVv8t5DfJffJln6ccZ+n0XaYCciDaf/jX7VjNq0El8I5Zz+VGC2DKQ4s47P2QGOJ9v7lu5MTbRRHgVYXQbM+yOaXb+Khlx5DlDEYtSoDZWHYUZPNCijCcQnLi9P3z0+hwEiWt6HwxVTSLMCqy5+9e3L85ivWl+CqCN5FUXASzr04CmFq6VYkvJhPmmsLJLYdHDVJtW/sxD5yI2cmZiCn+PoMsnLSKNYTY6Cgnr86PTmzl/Dm/fO6tNYPV3xojf1h3cwGSEdhEtVBGQC8m2ziwhQZAtBMRYqgBlUuQMOUDKhRCv0YYjL2q8GJRzQwDHxgCPhqVIT3gjjxKqdW2SF+9+r4xF6uk9+vvzh9qiGa8zjSMKjpCHZOpD3Ufvv25IWx/XhykRrRdCwztReRG6Wm9l7keZ+S0kr4JGs0XGDV86d/pGa1xjOL0PTvQySjpvZ9GHPmm9prD6ldhMU7mlpuDX04Dyx+Sr8ccV6OGLczWZKXeKWNCmCkGBTVIysafgLkm6e/BVQqontU2vYD1C3mCZxr17WS1Afo2ftTTY1++a9Q6z5DRRSUyPAjt/2h11IknL55lrG11/q2AtX50O6VoILZdQbZ7n0raCgpR78uk0kN2q9NZlAP4Cq0pF9/zaYMYVUTY5grqwuIhqPXqPvDiISq2tBaDGg0oOFYCCGOOJpOvIu5i9e4dMBCq7KVHIGSMkpwPsEZgqknkh+k29h2/OUvDHqkciBtEsUhcnNGCOeey6MPyHS8kMVWMN0Fxj/QmFQqQUUONobldpH3l8Hw/rf/FBAQ54cp5VvMmoZjALwKEDRD4oiY1ORkfZXbfCz0iMd2yK+kv7R+4MMXp+/UuL5kId2ZeCzpp/GMmyRyHs85LOMKkfLpbDTisZhawUVk6FDgpm+9BfffUatSf83SCbUedHkdU8yYHcMYlBedIa7qXhjy+Ad1ZYPHl8IjlNFHs3Q6g09G4D6jLrktST979+JpMVggRgDhUI0pCFaQT45Pzp57fuA574u5EiUioGPY4qHomtt0wsFO7h2pprKwCguOJ6jp9HypCwfrcwLKXWaCKpyXOHxG77oxEONWkZGUQMQp9NZCXjnlmKZBzIMS2NvX705evwCWmJGXzoig9hqBw/WjSLPFKotCg9yZ1q8FAt0wUUzuGpbUUJ5tCLUsb4dgNKVO6JwfswCb6shSC2k1S9IidG2z02rltDupL1BVwqmODcytvAO4Yv4zFLgkOSECGqY2wTvkLCnfNnY25ZBX2+pkG49jr8zaF3h9yf0plLvdMneRji86vNuWD53ObsfJJIMAqtPi/AgjP4rKpvKaJxO9xB4UPPwFjwKextd6p4XjG2ZJ7EqpqEZjvl6+iFkZoj0exRblYpTGWgu7Iczm7atmJ5tDjQbWSzzy6AWhAiSndMIDr0ToS7wm0wmYfErCgWKNxD84chf/7O6a1l751LS8UN9ZWEL1LOuFM38NVwcYZqHlsCStkIgxaU9WwKZk4GToYM/uvkn/Ke8L2Hzb2Avu3nZ/nzFs2y4jwCIg2EGYgzOGqGa+bwZwN+qRCRSJvVwhoYicy7LF0TvMAPl8afRptOhi8CX0ad+kFFZqjsRGOTuhYnPmSd0+w8HyA8jsTYwXCaCe2aZLqYWYLGV3mBXjxKFn3EGoid9CCfT6L20I+d4waQqUzbGfjpp1YyDJEjuUNhYwxLrRLBRM0kRXQ7+kzqYmST61l6op0T8vpG6ed8xds3thFhKBNpltq2WS/C/MrD+hFnVc1+xaWNYzoQC0bDhy97HM6pnWAS2Vy6jMyJsUci1r7cGoAHHe2MXGYvFe61GL9gRGGJL4V67PexYlYrvY47xr9sxGm9aORr3eo0cmydYC0e2eOWJ+grWig1FaZ+2a5y0s68lV0nSwbk/sKxetzi8vBiBa2AC5b2LxS77QT89bF9I0qK0eoky9tk/P2xdiKCtbhU1YlnV63rkwrGDmp97Uvz5zmM9i/SWCpBesY+1eyOEy1l1BAo1WEDc6D16a9G+j+wDIyLwry3oXyoZKxQKGH108liGkT/q+CfKqhGJP7OyN9Hu0zlhu2eOhtUdt1lzDhrA13aBbB2s2deEX4fBib/FDFCOgkhDpCmQBr3HJT4Jpep3bznDDQmlvwoNCdY4gxfXIPlqCmMjyEvLSRvZydklBypVjsLNLkRKHigAgisDmAMkFf0ohmVIcInIwxPD0GiiGxbBBGYB/LanexVxQ0G+sRJnMoRnYZqyCwk/gzsG+ehPizywEM/WhsVpRLoXoDc5GixKfR7AErrMbG7bBfbvdETYtua846tiEaczTY3FhUApqfxAFIFhsmEkGJLK0bSCDl7bM9dhCT6xrEzmAFI10eeXJhUkAiXVjPGhbB00xk7KQEgJrFM0fZMEP2a2BCJgjYTa7ySepQ8dt7pcHBChbMwDHWjwUMMjmdWZIeCdKdG48cE3Hui5m1cjNwxxmHV4eCUkHvBO4IaVHd5rYFWl9bLtNJD7idURvD3ryRerY2zj6JEOZlKkuc51cAcsWQvVZAhOheJbaLXNutwZ3qfl3392LrBkS9GfwnBb1ZWBeMgmyC/UazB/aY4ul2H0IZUwKVjnQu3SQ0rRomD1Wv3Ki2e3fsazZJQXM5fQVygLYb/S6uIqqG5ZMHt+/PrU/frMU/BZJsZ4iKY1OI/pm6wxbh2PdWGl4+PLncDzzUWv97a/aN8v5FqD5l/+IEQB4cjiMj75ZQucA9BzZgKt3MP3lnzUavN42eFMZDD4+JGvWiVJL1J3J449y39LQSpakycd+vQ7zZ8l16Gi5KPkCFViij4wlJKCEEtvsinnIK3nqTPSRuYR4JpHbr788efKsXpgzCp7LlcMICAJVgyLqrFY76xslk+hKp3ucUuR2WOgmtmjdnNPUxYA0auT5XGQyAxTFukpxtWgkwQ3olchsrAkjuo2lgB8NhjFnlyvl7GnMkBRl8qQEA/IUTVpL9WjtOt3M1gv1GKujq2yEfp7QOXRCCJsWas0hhZT5Rgkr1SfH6hu/j5tdY62iOu1W6wEXuLnbzJCtvv0o9RQbwJaMpQxrKJGiORcjK8wUud4aQGlisJYQrmQQgvGIFYO7A1K0kSlH2zL8CIXbDFlXcIwdUNgIsQ9EcAE6EqZt23Vqj9U/f85eJ17dEJFClBprFcsZ3KyLvKqoRGQHHInYwdBx9kwwbzwJeZL0rV7PpM8sffnW6knOVbZW/bo6neirdwRPIElR5arGQmnTRyvEMrlTkdmTYAbrYSsIhZcJvNC6FuzOXdJ1ww5CVTcVg3CfVNBVR4C0KBEGRYkwILUT3faKd6rXlfLAF1KDRoBbPg/HKaSqMOXHf5KBvKaJ/BSZGYjrBfu8/o6DJOpmvUTFy2I8vPZi8fskgap79PQ7PhYjZ1FMXRhqYj33qXWPhxcK+h0fqpFj2VCipyhGcU0LmT9zmXqSUE9h33I7FoqO2dso4R/e1y8kkZUjJlGc6jozh4Z9pAvCZXB4Az9h0Xe4xsODA+Pbg4PG2uywMqvOX7gch1xOeSdjqYRw7oiVF7bgquX43vSJmKFoK7BkfBzaeWvGgXtKueoj6HV5f1CnLKvsOyRmGBy8k8P1+gecXRNQUYh9UBrqOCa5Ll2CGhV1KLd+hirXE8AZ5ZlELx5nT/3yEVESKKzyaCr6U9NEYCP3DLd7BgejLP42Z1JOaLMVqh1HBJOLAr3Z15/2PQVTnhT3QacIT5mPqwuGkfnx9AckEToBUzZRAaYLEuaFiQTPw7m84doIAOvuonBW4hag/rguvrCp9+t0nVcf4HhFZ+lwb7dlVJAXZJBzqEPX65sUFEa7vvvj+uEw/9jgiPKDY+YyTThCzdZmoYzlFs08i6DHYIUW8BtsG5qazzTRPqZBN4LBWvW+UIH1I9EmsmON+ITYwONppG5Oj+oPBdSpIFY3+tKvyFwdVbqoviH0U9n7VJng4PYAK9kGFNuvcJ74PpaK7zu0zCaQaMcnDFnF0D4alliaRuOxL9XAHFp0kwxFsOY4mcgrqjlq5RQqsUk4lz2j12xKzvoXpayB/SSO2TXgxK9ehBfjcfEMS+oXb8o1B999Fwj7AnaiRaQwcsQwxAAVBXLArN+vP8yK67EorrMU0lBRSGVc56jPafEFNbJ0/Tw0nQs4iY+H4nrby66lS1fM3yydFV3Vekg/w5W8ufxoWJ8i1BkiSVSNKooSMvyU+EruJMyYgbJHeZbwgmzjHsuyLpnGEIbvvlMP92ybGeoZJYjL38xS3eoaA2ZR+z7VDTH6KqRBS2wkhCTIYD+lQ+WPvL5Wh8qu17bDDb8LVpeqHuV1olCU7fcoaKrDSmZcJfb5xS1aVVWlK1Qps2SiRwYkiZeM2Cv76MqSn/FnTu/KuLWLcC9T1s+f1zcoUl+heBstBv1rdLh/XjxfFBwN7KNA7CKcgg12qIJglW+X68fVelu6KPfM6sxTlnjOZu63P9wfjkZmvp3Mykp/7iAHSk1r0+XTdPJD7KW8L4sRI4/KV1U2yVT2ypKd/jci6ekgtSXffUVWtsp7tsLf2XXVaKyX7lw2VTH7W4R1LcxCOJURHH4ymqLonrKxiL1FB9YeKhVQyYIaj2x3e5RDyJdB5m6SStQs7KPFtvAqMBnCjiPDXQ9mcppcT74Rxk/meCAguu7R6+KEdZOylP8jcow7eX0uv+66uI3XRMgyl18RNfz1ULb6mn2cn9inegskMyQKjXGU3pnQeJSNZqQ5FKrp+yjULj+7G6ZAU3tz0rEWpmxcLlCHvHxgtR5RQylL/Nd7UynAU+sa/948BHCvJxtC5eZSutYjKtyApOKcmciSSyezkBR4yH5NcIOC1e9mwZAqVtUKNPlPieHOTKEsC9FuKCUJpEaVJKwMfEsKl8UKlcUVgrxL6Peq71vFXw5CVXBjJfdx0nhtnywD2yklvVU6pJcI7XtfkRYPtsKU6SK3Xop4ii6Vmt+x8UbyvmXjDZj1jXfyMFLuZRRXlqo9ru5LRfVaAOYxyxKR4D18tjLFbTD0Ob/8o8xO+cav1E6R5nFn0+RXhtNlYJUimoxKgaVimi0uYkUwUo37r238DP6/ic4obA+qB1Bhl2jO9GayaTj5/bp80MWxTPihzImxW6tnVqekcYKcwN5ySw91ohD//btTve7RpyZN+uyEVrjRVUhlil135Id84oMUTAiydNLyzZBGXxPdcBnTluQmmfjawN7+hcHdLfWv/ahk/auVvF9zGkXI8Quzc1NbXBlTWHjG/ZTpqnZHqVEu4d1NX30b92nvHV20b8VGUDvqDYtrZdkd1cv94XrsjT+IfvOnRFTc4he8zDvDRUBAdZpfSL8ZEnOsOfNR0soeNoudiTePks+flyppW+sAb/l6buOTKfPXff5nlr/sKn/GVf1s68LArvILh22EqsNd8utEp6/6jDVwyUAoSOAl8DgIbVTF4VcWcap3Li1WwuaNekN2vyUuxaZfGjGL/otswpcC5oWVIE/XQUOl2166Sglssmz1OXX967iw0bz/GZY+KhswvRR9FEJ+Prp4+FE7FH/libp2lOUaFg49hdEY9DcoYvIj3FW5U8aEa12jXZBJ9N5TyH8dreVKc7R1f4n998I+4LrOOFnDW4awn+iwcKYSMxoVvKbCZG7/ngxfr8/rxufP9ZTuMohkdaVymwzndlkyAhiK4YWufmkfybWXt63FNqKrr2K02l+mEpS4tuvG1txClhAq31XfgaUZ+TxlU+IeXa6kGaOHQr+KD+w3mqU8xcBau0l261a5iclrJ2oNGZQ85X+GdNgUf3902JT/i4f/BaU93Xk="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
# Si ya habia una version cargada, se cambia por la nueva SIN perder los
# modelos que ya estan en la GPU (antes Colab seguia usando el codigo viejo).
_old = sys.modules.get("ap3d.engine")
_eng = getattr(_old, "_ENGINE", None)
_app = sys.modules.get("ap3d.app")
if _app is not None:
    for _k in ("inline", "web"):
        try:
            _app._SERVERS.pop(_k).close()
        except Exception:
            pass
    try:
        _app._SERVERS.pop("tunnel").terminate()
    except Exception:
        pass
for _m in [m for m in sys.modules if m == "ap3d" or m.startswith("ap3d.")]:
    del sys.modules[_m]
if _eng is not None:
    from ap3d import engine as _e
    _eng.__class__ = _e.Engine
    _eng._ensure()
    _e._ENGINE = _eng
    print("Código actualizado; los modelos siguen cargados en la GPU.")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias, TripoSG y Blender 4.5 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2, TripoSG y Blender 4.5 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
INSTALAR_TRIPOSG = True  # @param {type:"boolean"}
# @markdown TripoSG es el modelo abierto del equipo de **Tripo AI**: formas con más detalle y bordes limpios. Se suma como segundo motor; si no instala, sigue Hunyuan3D.
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25' sentencepiece protobuf")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender: 4.5 LTS (el mas nuevo estable); si no se puede, 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    for ver, serie in (("4.5.3", "4.5"), ("4.5.2", "4.5"), ("4.5.1", "4.5"), ("4.2.3", "4.2")):
        url = f"https://download.blender.org/release/Blender{serie}/blender-{ver}-linux-x64.tar.xz"
        r = subprocess.run(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz "
                           f"-C /content && mv /content/blender-{ver}-linux-x64 /content/blender",
                           shell=True)
        subprocess.run("rm -f /content/blender.tar.xz", shell=True)
        if r.returncode == 0 and os.path.exists("/content/blender/blender"):
            break

# --- TripoSG (modelo abierto del equipo de Tripo AI): forma de alta fidelidad
if INSTALAR_TRIPOSG and not os.path.exists("/content/TripoSG"):
    sh("git clone -q --depth 1 https://github.com/VAST-AI-Research/TripoSG /content/TripoSG", check=False)
    sh(f"{sys.executable} -m pip install -q peft jaxtyping typeguard scikit-image", check=False)
    r = subprocess.run(f"{sys.executable} -m pip install -q diso", shell=True,
                       capture_output=True, text=True)
    print("TripoSG:", "listo" if r.returncode == 0 else "diso no compilo (TripoSG desactivado)")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
MODO_CALIDAD_TAMBIEN = True  # @param {type:"boolean"}
CARGAR_TRIPOSG = True  # @param {type:"boolean"}
CARGAR_TEXTO_A_3D = True  # @param {type:"boolean"}
# @markdown Turbo (5 pasos + FlashVDM) siempre se carga: la forma sale en segundos. Con la casilla de arriba también queda cargado el modelo de 50 pasos para el preajuste "Maxima".
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

# ---- reloj de pasos: duracion de cada uno y fin estimado ----
_T0 = time.time()
_RES = []
_EST = {"Hunyuan3D-2": 90, "Hunyuan3D-2mv": 45, "TripoSG": 60, "Texto a imagen (FLUX)": 150,
        "Precargar pesos en RAM": 45, "Cargar modelos en GPU": 100, "TripoSG en GPU": 25}
_PEND = [k for k in _EST]
def paso(nombre, fn, activo=True):
    if nombre in _PEND:
        _PEND.remove(nombre)
    if not activo:
        _RES.append((nombre, 0, "omitido"))
        return None
    resto = sum(_EST[k] for k in _PEND)
    print(f"[{(time.time()-_T0)/60:4.1f} min] ▶ {nombre}… (despues faltan ~{resto/60:.0f} min)", flush=True)
    t = time.time()
    try:
        r = fn()
        _RES.append((nombre, time.time() - t, "ok"))
        print(f"           ✓ {nombre} en {time.time()-t:.0f} s", flush=True)
        return r
    except Exception as e:
        _RES.append((nombre, time.time() - t, f"fallo: {str(e)[:120]}"))
        print(f"           ⚠ {nombre}: {e}", flush=True)
        return None

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))
print(f"Inicio. Primera vez: unos {sum(_EST.values())/60:.0f} min (despues de reiniciar, mucho menos: ya esta descargado).")

pats = ["hunyuan3d-dit-v2-0-turbo/*"] + (["hunyuan3d-dit-v2-0/*"] if MODO_CALIDAD_TAMBIEN else [])
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
paso("Hunyuan3D-2", lambda: snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16))
paso("Hunyuan3D-2mv", lambda: snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv-turbo/*"]
     + (["hunyuan3d-dit-v2-mv/*"] if MODO_CALIDAD_TAMBIEN else []), max_workers=16), CARGAR_MULTIVISTA)
def _tripo():
    snapshot_download("VAST-AI/TripoSG", local_dir="/content/TripoSG/pretrained_weights/TripoSG", max_workers=16)
    snapshot_download("briaai/RMBG-1.4", local_dir="/content/TripoSG/pretrained_weights/RMBG-1.4", max_workers=16)
    return True
CARGAR_TRIPOSG = bool(paso("TripoSG", _tripo, CARGAR_TRIPOSG and os.path.exists("/content/TripoSG")))
paso("Texto a imagen (FLUX)", lambda: snapshot_download(
     "black-forest-labs/FLUX.1-schnell", max_workers=16,
     ignore_patterns=["flux1-schnell.safetensors", "ae.safetensors", "*.md"]), CARGAR_TEXTO_A_3D)

eng = get_engine()
paso("Precargar pesos en RAM", eng.preload_weights, P["preload_weights"])
paso("Cargar modelos en GPU", lambda: eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA,
     want_turbo=True, want_quality=MODO_CALIDAD_TAMBIEN))
paso("TripoSG en GPU", eng.load_triposg, CARGAR_TRIPOSG)

print("\n======== RESUMEN (mandame esto para revisar) ========")
for n, sec, st in _RES:
    print(f"{n:28s} {sec:6.0f} s   {'✅' if st == 'ok' else ('—' if st == 'omitido' else '⚠️')} {st}")
print(f"{'TOTAL':28s} {time.time()-_T0:6.0f} s")
print(f"GPU {P['gpu']} · VRAM usada {torch.cuda.memory_allocated()/2**30:.1f} GB · "
      f"TripoSG {'si' if getattr(eng, 'tripo', None) is not None else 'no'} · "
      f"texto a 3D {'si' if CARGAR_TEXTO_A_3D else 'no'}")
print("Todo listo. Sigue con la celda 3b (prueba de 1 min) o con la 4/5.")

## 3b. Prueba rápida sin página web (opcional)

In [ ]:
# @title 3b. Prueba corta SIN página web (~1-2 min): confirma que todo funciona y mide tiempos
import glob, sys, time
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
_img = sorted(glob.glob("/content/Hunyuan3D-2/assets/example_images/*.png"))[0]
_t = time.time()
_last = [None]
def _p(f, m):
    if m != _last[0]:
        _last[0] = m
        print(f"[{time.time()-_t:5.0f} s] {int(f*100):3d}%  {m}", flush=True)
_r = get_engine().run(_img, steps=5, octree=384, faces=60000, tex=2048, rig="auto",
                      tex_rapida=True, progress=_p)
_i = _r["info"]
print(f"\n======== RESULTADO (mandame esto para revisar) ========")
print(f"Listo en {time.time()-_t:.0f} s  ->  {_r['zip']}")
print(f"Tiempos: forma {_i.get('segundos_forma')} s · textura {_i.get('segundos_textura')} s · "
      f"Blender {_i.get('segundos_blender')} s")
print(f"Detectado: {(_i.get('clasificacion') or {}).get('tipo', _i.get('rig'))} · huesos {_i.get('huesos')} · "
      f"pesos por {_i.get('pesos')} · malla IA {_i.get('caras_alta')} -> {_i.get('caras')} caras · "
      f"cuadros {_i.get('quads_caras')}")
print("Archivos:", ", ".join(sorted((_i.get('archivos') or {}).values())))
print("Avisos:", _i.get("avisos") or "ninguno")

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()